# NeoMint-3B: LoRA Fine-Tuning with Unsloth

This notebook fine-tunes **Qwen2.5-3B-Instruct** using LoRA (via Unsloth) on the NeoMint OS task instruction dataset.

**What this does:**
1. Installs Unsloth and dependencies
2. Loads the base model (Qwen2.5-3B-Instruct) with 4-bit quantization
3. Applies LoRA adapters to attention + MLP layers
4. Trains on the NeoMint instruction dataset
5. Exports to GGUF Q4_K_M format for Ollama deployment

**Runtime:** Google Colab with T4 GPU (free tier works)

**Expected training time:** ~15-30 minutes on T4

## 1. Environment Setup

In [ ]:
# Verify GPU availability
!nvidia-smi

In [ ]:
%%capture
# Install Unsloth (optimized LoRA fine-tuning library)
!pip install unsloth
# Unsloth pins its own versions of transformers, peft, trl, etc.
# No need to install them separately

In [ ]:
import json
import torch
from pathlib import Path

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## 2. Configuration

All hyperparameters in one place for easy tuning.

In [ ]:
# ── Model config ──────────────────────────────────────────
BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

# ── LoRA config ───────────────────────────────────────────
LORA_R = 32                  # LoRA rank (higher = more capacity, more VRAM)
LORA_ALPHA = 64              # Scaling factor (typically 2x rank)
LORA_DROPOUT = 0.05          # Dropout for regularization
TARGET_MODULES = [           # Which layers to apply LoRA to
    "q_proj", "k_proj", "v_proj", "o_proj",  # attention
    "gate_proj", "up_proj", "down_proj",       # MLP
]

# ── Training config ───────────────────────────────────────
NUM_EPOCHS = 3
BATCH_SIZE = 2               # Per-device batch size
GRADIENT_ACCUMULATION = 4    # Effective batch = BATCH_SIZE * GRADIENT_ACCUMULATION = 8
LEARNING_RATE = 2e-4
WARMUP_STEPS = 10
WEIGHT_DECAY = 0.01
LR_SCHEDULER = "cosine"
SEED = 42

# ── Paths ─────────────────────────────────────────────────
DATASET_PATH = "neomint_train.jsonl"  # Upload this to Colab
OUTPUT_DIR = "./neomint-3b-lora"
GGUF_OUTPUT = "./neomint-3b-Q4_K_M.gguf"

print("Config loaded.")
print(f"  Base model:     {BASE_MODEL}")
print(f"  LoRA rank:      {LORA_R}")
print(f"  Epochs:         {NUM_EPOCHS}")
print(f"  Effective batch: {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"  Learning rate:  {LEARNING_RATE}")

## 3. Load Base Model + LoRA Adapters

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,  # auto-detect (float16 on T4)
)

print(f"Model loaded: {model.config._name_or_path}")
print(f"Parameters: {model.num_parameters():,}")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized checkpointing
    random_state=SEED,
)

trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 4. Load and Format Dataset

The NeoMint dataset uses the Alpaca instruction format:
```json
{"instruction": "...", "input": "...", "output": "..."}
```

We convert it to chat-style messages for Qwen2.5-Instruct.

In [ ]:
from google.colab import files
import os

# Upload dataset if not already present
if not os.path.exists(DATASET_PATH):
    print("Please upload neomint_train.jsonl:")
    uploaded = files.upload()
    for fname in uploaded:
        if fname.endswith(".jsonl"):
            os.rename(fname, DATASET_PATH)
            print(f"Saved as {DATASET_PATH}")
else:
    print(f"Dataset already present: {DATASET_PATH}")

# Load and count
with open(DATASET_PATH, "r") as f:
    raw_data = [json.loads(line) for line in f]
print(f"Loaded {len(raw_data)} training examples")

In [ ]:
SYSTEM_PROMPT = """You are NeoMint, an AI assistant integrated into a Linux Mint operating system. You help users interact with their OS through natural language.

You have access to these tools:
- list_files(path): List directory contents
- read_file(path): Read a file's text content
- write_file(path, content): Write/overwrite a file
- run_command(cmd): Run a shell command (validated against an allowlist)
- open_application(app_name): Launch a GUI application by name
- get_clipboard(): Return current clipboard text
- set_clipboard(text): Set clipboard text

When you need to use a tool, output a tool_call block:
```tool_call
{"tool": "tool_name", "args": {"arg1": "value1"}}
```

Rules:
1. Use tools when a task requires OS interaction
2. Explain what you're doing before calling a tool
3. If a request is ambiguous, ask for clarification
4. For multi-step tasks, execute one step at a time
5. Never execute dangerous commands without user confirmation"""


def format_as_chat(example: dict) -> str:
    """Convert an Alpaca-style example to Qwen2.5 chat format."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    # Build user message from instruction + optional input context
    user_msg = example["instruction"]
    if example.get("input", "").strip():
        user_msg += f"\n\nContext: {example['input']}"

    messages.append({"role": "user", "content": user_msg})
    messages.append({"role": "assistant", "content": example["output"]})

    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )


# Format all examples
formatted_data = [{"text": format_as_chat(ex)} for ex in raw_data]

# Preview one example
print("=" * 60)
print("SAMPLE FORMATTED ENTRY:")
print("=" * 60)
print(formatted_data[0]["text"][:800])
print("...")

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(formatted_data)
print(f"Dataset: {dataset}")
print(f"Features: {dataset.features}")

# Check token lengths
token_lengths = [len(tokenizer.encode(ex["text"])) for ex in formatted_data]
print(f"\nToken length stats:")
print(f"  Min:  {min(token_lengths)}")
print(f"  Max:  {max(token_lengths)}")
print(f"  Mean: {sum(token_lengths)/len(token_lengths):.0f}")
print(f"  Entries > {MAX_SEQ_LENGTH} tokens: {sum(1 for t in token_lengths if t > MAX_SEQ_LENGTH)}")

## 5. Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,  # Pack multiple short examples into one sequence for efficiency
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        weight_decay=WEIGHT_DECAY,
        lr_scheduler_type=LR_SCHEDULER,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        save_strategy="epoch",
        seed=SEED,
        report_to="none",
        optim="adamw_8bit",
    ),
)

print(f"Training {NUM_EPOCHS} epochs on {len(dataset)} examples...")
print(f"Packing enabled — multiple examples per sequence for efficiency")

In [ ]:
# Show GPU memory before training
gpu_stats = torch.cuda.get_device_properties(0)
reserved = torch.cuda.max_memory_reserved() / 1024**3
print(f"GPU: {gpu_stats.name} ({gpu_stats.total_mem / 1024**3:.1f} GB)")
print(f"Reserved before training: {reserved:.2f} GB")

# Train
trainer_stats = trainer.train()

# Show results
peak_mem = torch.cuda.max_memory_reserved() / 1024**3
print(f"\n{'='*60}")
print(f"Training complete!")
print(f"  Total steps:    {trainer_stats.global_step}")
print(f"  Training loss:  {trainer_stats.training_loss:.4f}")
print(f"  Runtime:        {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"  Peak GPU mem:   {peak_mem:.2f} GB")

## 6. Test the Fine-Tuned Model

In [ ]:
# Switch to inference mode
FastLanguageModel.for_inference(model)


def test_prompt(instruction: str) -> str:
    """Send a test instruction and get the model's response."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": instruction},
    ]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )
    return response


# Run test prompts
test_cases = [
    "List all files in my Documents folder",
    "How much disk space do I have left?",
    "Open Firefox",
    "Read my notes.txt and save a summary",
    "What's on my clipboard?",
]

for prompt in test_cases:
    print(f"\n{'='*60}")
    print(f"USER: {prompt}")
    print(f"{'='*60}")
    response = test_prompt(prompt)
    print(f"NEOMINT: {response}")

## 7. Save LoRA Adapters

In [ ]:
# Save the LoRA adapters (small — typically 50-100 MB)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapters saved to: {OUTPUT_DIR}")

# Check adapter size
import os
total_size = sum(
    os.path.getsize(os.path.join(OUTPUT_DIR, f))
    for f in os.listdir(OUTPUT_DIR)
    if os.path.isfile(os.path.join(OUTPUT_DIR, f))
)
print(f"Adapter size: {total_size / 1024**2:.1f} MB")

## 8. Export to GGUF Q4_K_M

This merges the LoRA adapters back into the base model and exports to GGUF format
with Q4_K_M quantization — optimized for CPU inference via Ollama.

In [ ]:
# Export to GGUF — Unsloth handles the merge + conversion in one call
model.save_pretrained_gguf(
    "neomint-3b",
    tokenizer,
    quantization_method="q4_k_m",
)

print(f"\nGGUF export complete!")
print(f"Look for: neomint-3b-unsloth-Q4_K_M.gguf")

In [ ]:
# Check the GGUF file size
import glob

gguf_files = glob.glob("neomint-3b*.gguf")
for f in gguf_files:
    size_gb = os.path.getsize(f) / 1024**3
    print(f"{f}: {size_gb:.2f} GB")

print(f"\nExpected size: ~1.8 GB for Q4_K_M of a 3B model")

## 9. Download the GGUF File

Download the GGUF file from Colab. You'll load this into Ollama on your Linux Mint machine.

In [ ]:
from google.colab import files

gguf_files = glob.glob("neomint-3b*.gguf")
if gguf_files:
    print(f"Downloading: {gguf_files[0]}")
    files.download(gguf_files[0])
else:
    print("No GGUF file found. Check the export step above.")

## 10. Training Summary

Save training metadata for the capstone paper.

In [ ]:
summary = {
    "base_model": BASE_MODEL,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "target_modules": TARGET_MODULES,
    "num_epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation": GRADIENT_ACCUMULATION,
    "effective_batch_size": BATCH_SIZE * GRADIENT_ACCUMULATION,
    "learning_rate": LEARNING_RATE,
    "warmup_steps": WARMUP_STEPS,
    "lr_scheduler": LR_SCHEDULER,
    "max_seq_length": MAX_SEQ_LENGTH,
    "dataset_size": len(raw_data),
    "training_loss": trainer_stats.training_loss,
    "training_runtime_seconds": trainer_stats.metrics["train_runtime"],
    "total_steps": trainer_stats.global_step,
    "peak_gpu_memory_gb": round(torch.cuda.max_memory_reserved() / 1024**3, 2),
    "quantization": "Q4_K_M",
    "gguf_files": gguf_files,
}

with open("training_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print(f"\nSaved to training_summary.json")

# Download the summary too
files.download("training_summary.json")

---

## Next Steps

1. Copy the GGUF file to your Linux Mint machine
2. Create the Ollama Modelfile (see `neomint.modelfile` in the repo root)
3. Run: `ollama create neomint-3b -f neomint.modelfile`
4. Test: `ollama run neomint-3b`

See `fine-tuning/README.md` for detailed instructions.